# BioNeMo NIM Deployment Guide - Mini Hands-On

Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

This notebook describes the deployment path for AI-Powered-Drug-Discovery-Bootcamp. In this bootcamp, the preferred x86 cluster path is Apptainer/Singularity rather than Docker. On ARM64 nodes, run Boltz-2 locally and point `MOLMIM_URL` to a hosted MolMIM endpoint. The wrapper scripts at the repository root write the actual endpoints to `.openhackathon-nims.env` for notebooks to load automatically.


### Setup NGC API Key

The first step in deploying BioNeMo NIM endpoints is configuring your environment to access [NVIDIA NGC](https://catalog.ngc.nvidia.com/). Set `NGC_API_KEY` in the terminal where you launch the services.

```bash
export NGC_API_KEY=<PASTE_API_KEY_HERE>
```

The notebooks do not print or store this key. The service wrapper passes it to the NIM containers when pulling or starting the images.


### Self-Hosted NIMs on the Deployment Cluster


On the deployment cluster, use Apptainer/Singularity through the repository wrapper instead of Docker. From the repository root:

```bash
scripts/openhackathon_services.sh start --boltz2 1
source .openhackathon-nims.env
scripts/openhackathon_services.sh status
```

The wrapper selects available ports, starts MolMIM and one or more Boltz-2 endpoints on x86, or preserves hosted `MOLMIM_URL` while starting local Boltz-2 on ARM64. It records `MOLMIM_URL`, `BOLTZ2_URL`, and `BOLTZ2_ENDPOINTS` in `.openhackathon-nims.env`.


### Verify the Endpoints

After sourcing `.openhackathon-nims.env`, verify health from the repository root:

```bash
scripts/check_nim_health.sh
```

You can also check manually:

```bash
scripts/check_nim_health.sh molmim "$MOLMIM_URL" 1
scripts/check_nim_health.sh boltz2 "$BOLTZ2_URL" 1
```


### Cache and Image Notes

The Apptainer scripts keep NIM image/cache paths outside the notebooks. If your cluster requires a specific cache location, set it before starting services, for example:

```bash
export LOCAL_NIM_CACHE="$HOME/.cache/nim"
mkdir -p "$LOCAL_NIM_CACHE"
scripts/openhackathon_services.sh start --boltz2 1
```

See [`../singularity.md`](../singularity.md), [`../deployment.md`](../deployment.md), and [`../scripts/run_nim_container.sh`](../scripts/run_nim_container.sh) for the full deployment details.


### Multiple Boltz-2 Endpoints

If you have multiple GPUs allocated, you can start one Boltz-2 endpoint per GPU:

```bash
scripts/openhackathon_services.sh start --boltz2 2
source .openhackathon-nims.env
```

The design pipeline reads `BOLTZ2_ENDPOINTS` and can distribute affinity predictions across those endpoints.


### Notebook Environment

Before opening JupyterLab, install the Python dependencies and source the generated endpoint file:

```bash
pip install -r deployment-requirements.txt
source .openhackathon-nims.env
jupyter-lab mini-hands-on/00_Introduction.ipynb
```

If JupyterLab was already running before the services started, either restart the kernel or rely on the notebook helper code that reads `.openhackathon-nims.env` automatically.


### Stop Services

When finished, stop the NIM services from the repository root:

```bash
scripts/openhackathon_services.sh stop
```


### Docker Workstations

If you are running on a workstation with Docker instead of the AI-Powered-Drug-Discovery-Bootcamp deployment cluster, use [`../deployment.md`](../deployment.md) as the alternate deployment path.


## Additional Tools

In addition to the BioNeMo NIM inference endpoints, the hands-on notebooks use the repository's shared CDK oracle and scoring utilities.

### Boltz2-Python-Client

The Boltz-2 Python client provides synchronous and asynchronous interfaces for NVIDIA's Boltz-2 biomolecular structure prediction service. The notebooks are compatible with the current client format used by `boltz2-python-client>=0.5.2`.

- https://pypi.org/project/boltz2-python-client/
- https://github.com/NVIDIA/digital-biology-examples/tree/main/examples/nims/boltz-2

### ReaSyn with MCP Server

The mini-track includes an optional ReaSyn MCP workflow for retrosynthesis. In this integrated mini-track, the ReaSyn section is optional and skipped unless `fastmcp` is installed and `REASYN_URL` points at a running ReaSyn MCP server.
